In [ ]:
import tensorflow as tf
import random
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from matplotlib import rc
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

print(tf.__version__)

rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

seed = 42
random.seed(seed)
tf.random.set_seed(seed)
np.random.seed(seed)

### 트레인셋과 테스트셋에서 target 열의 클래스 비율이 유지되도록 8:2 split (함수)

In [ ]:
def split_data(data):
    # feature 컬럼만 가져오기
    X = data.drop(['Target','SurgeryID', 'PatientID', 'SurgeryDate', 'BodyPart', 'SideEffect'], axis=1)

    # target 컬럼만 가져오기
    y = data['Target']
    
    # train/test 데이터 분리 (8:2)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

    return X_train, X_test, y_train, y_test


### Data Scaling z-score (함수)

In [ ]:
def data_scaling(X_train, X_test):
   X_columns = X_train.columns
   stds = StandardScaler()
   stds.fit(X_train)
   X_train = stds.transform(X_train)
   X_test = stds.transform(X_test)

   X_train = pd.DataFrame(X_train, columns=X_columns)
   X_test = pd.DataFrame(X_test, columns=X_columns)

   return X_train, X_test


### Random Forest 모델 학습 및 성능 평가, 시각화 (함수)

In [ ]:
def conf_matrix(y_test, y_pred, aftereffect, save_dir):
    # plt.figure(figsize = (7, 7))
    # y_test와 results의 numpy 행렬에서 최고 차수(-1)를 기준으로 비교하여 최고값을 선정.
    # cm = confusion_matrix(np.argmax(y_test, axis = -1), np.argmax(results, axis = -1))
    y_test = np.argmax(y_test, axis=-1).tolist()
    y_pred = np.argmax(y_pred, axis=-1).tolist()
    
    # 혼동행렬 생성
    cm = confusion_matrix(y_test, y_pred)

    # sklearn의 데이터를 이용하여 heatmap을 시각화해주는 seaborn의 패키지
    sns.heatmap(cm, annot = True, fmt = 'd',cmap = 'Blues')

    plt.title(aftereffect)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')

    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(save_dir+aftereffect+'_'+'CMatrix.png')
    plt.show()

    print('\n', classification_report(y_test, y_pred))


In [ ]:
data_path = '../dataset/dataset_12v/belly/'
sub_path = '.xlsx'
save_dir = '../result/dataset_12v/belly/'

aftereffect_list = [
    '장액종',
    '멍',
    '뭉침',
    '울퉁불퉁'
]

### 후유증별 타켓 분포 확인

In [ ]:
for val in aftereffect_list:
    path = data_path + val + sub_path
    df = pd.read_excel(path)
    
    print(val, f"({df.shape[0]}건)")
    print(df['Target'].value_counts(), "\n")

## Random Forest 이용

In [ ]:
for val in aftereffect_list:
    path = data_path + val + sub_path
    print(path)
    data = pd.read_excel(path)
    
    # KPI 데이터만 사용
    # data = data[selected_columns]

    # 8:2 split
    X_train, X_test, y_train, y_test = split_data(data)

    # 테스트셋 저장
    test_data = pd.concat([X_test, y_test], axis=1)
    os.makedirs(save_dir, exist_ok=True)
    test_data.to_excel(os.path.join(save_dir, f'{val}_dataset.xlsx'), index=False)

    # 타겟 클래스를 one-hot 벡터로 변환
    y_train = keras.utils.to_categorical(y_train)
    y_test = keras.utils.to_categorical(y_test)
    
    # 데이터 스케일링
    # data_scaling(X_train, X_test)

    # RandomForestClassifier 모델 객체 생성
    RF_model = RandomForestClassifier(n_estimators=100, random_state=seed)

    # 모든 특징점을 사용하여 모델을 학습시키고, 테스트 데이터에서 성능을 측정
    RF_model.fit(X_train, y_train)

    # 훈련 세트와 검증 세트의 성능 지표 계산
    train_acc = accuracy_score(y_train, RF_model.predict(X_train))
    test_acc = accuracy_score(y_test, RF_model.predict(X_test))

    print("Accuracy with All Features")
    print(f"Train accuracy: {train_acc:.5f}")
    print(f"Test accuracy: {test_acc:.5f}")

    # 모델 저장
    os.makedirs(save_dir, exist_ok=True)
    joblib.dump(RF_model, os.path.join(save_dir, f'{val}_RFmodel.pkl'))
    # RF_model.save(os.path.join(save_dir, '{val}_RFmodel.h5'))
    
    # 검증 세트 예측
    y_pred = RF_model.predict(X_test)

    # 혼동행렬 시각화 및 성능지표 출력
    conf_matrix(y_test, y_pred, val, save_dir)
